In [1]:
import pandas as pd
import glob

# ======================================================
# PART 1 — CLEAN SARI WORKSHOP DATA
# ======================================================

file_paths = glob.glob("AD*.xlsx")

cleaned_dfs = []

for f in file_paths:
    df = pd.read_excel(f)

    # --- Clean column names ---
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace(" ", "_")
        .str.replace(".", "", regex=False)
        .str.lower()
    )

    # --- Detect and fix date column ---
    date_cols = [col for col in df.columns if "date" in col]
    if not date_cols:
        continue

    df["date"] = pd.to_datetime(df[date_cols[0]], errors="coerce")
    df = df[df["date"].notna()]

    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month

    # --- Remove blank rows ---
    if "name_of_school" in df.columns:
        df = df[df["name_of_school"].notna()]
    if "no_participants" in df.columns:
        df = df[df["no_participants"].notna()]

    # --- Extract Province ---
    province_cols = ["leinster", "munster", "connacht", "ulster"]
    province_cols_existing = [col for col in province_cols if col in df.columns]

    if province_cols_existing:
        def get_province(row):
            for col in province_cols_existing:
                value = row[col]
                if pd.notna(value) and str(value).strip() not in ["0", "0.0", "nan", ""]:
                    return col.capitalize()
            return None

        df["province"] = df.apply(get_province, axis=1)
        df = df.drop(columns=province_cols_existing)
    else:
        df["province"] = None

    # --- Identify core columns ---
    core_cols = [
        "date", "year", "month",
        "name_of_school", "location",
        "no_sessions", "no_participants",
        "province"
    ]

    core_cols_existing = [col for col in core_cols if col in df.columns]

    nationality_cols = [col for col in df.columns if col not in core_cols_existing]

    # --- Convert nationality columns to numeric ---
    df[nationality_cols] = df[nationality_cols].apply(
        pd.to_numeric, errors="coerce"
    )
    df[nationality_cols] = df[nationality_cols].fillna(0)

    # --- Compute Diversity Count ---
    df["diversity_count"] = (df[nationality_cols] > 0).sum(axis=1)

    df = df.drop(columns=nationality_cols)

    cleaned_dfs.append(df)

master_df = pd.concat(cleaned_dfs, ignore_index=True)

# --- Keep only 2023–2025 and valid provinces ---
master_df = master_df[
    (master_df["year"].isin([2023, 2024, 2025])) &
    (master_df["province"].notna())
]

print("Final SARI dataset shape:", master_df.shape)
print(master_df["province"].value_counts())

# ======================================================
# PART 2 — LOAD CSO POPULATION DATA
# ======================================================

cso_df = pd.read_csv("IPEADS02.20260302T030301.csv")

cso_clean = cso_df[
    (cso_df["Sex"] == "Both sexes") &
    (cso_df["Year"] == 2023)
][["County and City", "VALUE"]].copy()

cso_clean.columns = ["county", "population"]

cso_clean = cso_clean[cso_clean["county"] != "Ireland"]

# Clean county names
cso_clean["county"] = (
    cso_clean["county"]
    .str.replace(" County Council", "", regex=False)
    .str.replace(" City Council", "", regex=False)
    .str.replace(" City", "", regex=False)
    .str.strip()
)

# ======================================================
# PART 3 — MAP COUNTIES TO PROVINCES
# ======================================================

province_map = {
    # Leinster
    "Dublin": "Leinster",
    "Fingal": "Leinster",
    "Dún Laoghaire Rathdown": "Leinster",
    "South Dublin": "Leinster",
    "Kildare": "Leinster",
    "Meath": "Leinster",
    "Wicklow": "Leinster",
    "Louth": "Leinster",
    "Westmeath": "Leinster",
    "Offaly": "Leinster",
    "Laois": "Leinster",
    "Longford": "Leinster",
    "Carlow": "Leinster",
    "Kilkenny": "Leinster",
    "Wexford": "Leinster",

    # Munster
    "Cork": "Munster",
    "Kerry": "Munster",
    "Limerick": "Munster",
    "Clare": "Munster",
    "Tipperary": "Munster",
    "Waterford": "Munster",

    # Connacht
    "Galway": "Connacht",
    "Mayo": "Connacht",
    "Roscommon": "Connacht",
    "Sligo": "Connacht",
    "Leitrim": "Connacht",

    # Ulster
    "Donegal": "Ulster",
    "Cavan": "Ulster",
    "Monaghan": "Ulster"
}

cso_clean["province"] = cso_clean["county"].map(province_map)
cso_clean = cso_clean[cso_clean["province"].notna()]

province_population = (
    cso_clean.groupby("province")["population"]
    .sum()
    .reset_index()
)

print("\nProvince Population:")
print(province_population)

# ======================================================
# PART 4 — EQUITY CALCULATION
# ======================================================

workshops_by_province = (
    master_df.groupby("province")
    .size()
    .reset_index(name="workshops")
)

equity_df = workshops_by_province.merge(
    province_population,
    on="province",
    how="left"
)

equity_df["workshops_per_100k"] = (
    equity_df["workshops"] / equity_df["population"]
) * 100000

print("\nEquity Table:")
print(equity_df)


C:\Users\New User\AppData\Local\Temp\ipykernel_24516\2155160035.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["diversity_count"] = (df[nationality_cols] > 0).sum(axis=1)
C:\Users\New User\AppData\Local\Temp\ipykernel_24516\2155160035.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["diversity_count"] = (df[nationality_cols] > 0).sum(axis=1)
C:\Users\New User\AppData\Local\Temp\ipykernel_24516\2155160035.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inser

Final SARI dataset shape: (172, 9)
province
Leinster    141
Ulster       13
Munster      13
Connacht      5
Name: count, dtype: int64

Province Population:
   province  population
0  Connacht      621612
1  Leinster     3048854
2   Munster     1089576
3    Ulster      336076

Equity Table:
   province  workshops  population  workshops_per_100k
0  Connacht          5      621612            0.804360
1  Leinster        141     3048854            4.624688
2   Munster         13     1089576            1.193125
3    Ulster         13      336076            3.868173


C:\Users\New User\AppData\Local\Temp\ipykernel_24516\2155160035.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["diversity_count"] = (df[nationality_cols] > 0).sum(axis=1)


In [2]:
participants_by_province = (
    master_df.groupby("province")["no_participants"]
    .sum()
    .reset_index(name="participants")
)

equity_df = equity_df.merge(
    participants_by_province,
    on="province",
    how="left"
)

equity_df["participants_per_100k"] = (
    equity_df["participants"] / equity_df["population"]
) * 100000

print(equity_df)


   province  workshops  population  workshops_per_100k  participants  \
0  Connacht          5      621612            0.804360         308.0   
1  Leinster        141     3048854            4.624688        8506.0   
2   Munster         13     1089576            1.193125         752.0   
3    Ulster         13      336076            3.868173         729.0   

   participants_per_100k  
0              49.548593  
1             278.990073  
2              69.017673  
3             216.915222  


In [3]:
# Workshops per province per year
yearly_counts = (
    master_df.groupby(["year", "province"])
    .size()
    .reset_index(name="workshops")
)

# Merge with province population
yearly_equity = yearly_counts.merge(
    province_population,
    on="province",
    how="left"
)

# Compute per 100k per year
yearly_equity["workshops_per_100k"] = (
    yearly_equity["workshops"] / yearly_equity["population"]
) * 100000

print(yearly_equity.sort_values(["year", "workshops_per_100k"], ascending=[True, False]))


   year  province  workshops  population  workshops_per_100k
2  2023    Ulster          8      336076            2.380414
1  2023  Leinster         41     3048854            1.344768
0  2023  Connacht          4      621612            0.643488
3  2024  Leinster         30     3048854            0.983976
4  2024   Munster         10     1089576            0.917788
5  2024    Ulster          3      336076            0.892655
7  2025  Leinster         70     3048854            2.295945
9  2025    Ulster          2      336076            0.595103
8  2025   Munster          3     1089576            0.275336
6  2025  Connacht          1      621612            0.160872


In [4]:
# Define full structure
years = [2023, 2024, 2025]
provinces = ["Leinster", "Munster", "Connacht", "Ulster"]

full_index = pd.MultiIndex.from_product(
    [years, provinces],
    names=["year", "province"]
)

# Compute grouped counts
yearly_counts = (
    master_df.groupby(["year", "province"])
    .size()
    .reindex(full_index, fill_value=0)
    .reset_index(name="workshops")
)

# Merge with population
yearly_equity = yearly_counts.merge(
    province_population,
    on="province",
    how="left"
)

# Compute per 100k
yearly_equity["workshops_per_100k"] = (
    yearly_equity["workshops"] / yearly_equity["population"]
) * 100000

print(yearly_equity.sort_values(["year", "province"]))


    year  province  workshops  population  workshops_per_100k
2   2023  Connacht          4      621612            0.643488
0   2023  Leinster         41     3048854            1.344768
1   2023   Munster          0     1089576            0.000000
3   2023    Ulster          8      336076            2.380414
6   2024  Connacht          0      621612            0.000000
4   2024  Leinster         30     3048854            0.983976
5   2024   Munster         10     1089576            0.917788
7   2024    Ulster          3      336076            0.892655
10  2025  Connacht          1      621612            0.160872
8   2025  Leinster         70     3048854            2.295945
9   2025   Munster          3     1089576            0.275336
11  2025    Ulster          2      336076            0.595103


In [5]:
line_df = yearly_equity[[
    "year",
    "province",
    "workshops_per_100k"
]].copy()

print(line_df)


    year  province  workshops_per_100k
0   2023  Leinster            1.344768
1   2023   Munster            0.000000
2   2023  Connacht            0.643488
3   2023    Ulster            2.380414
4   2024  Leinster            0.983976
5   2024   Munster            0.917788
6   2024  Connacht            0.000000
7   2024    Ulster            0.892655
8   2025  Leinster            2.295945
9   2025   Munster            0.275336
10  2025  Connacht            0.160872
11  2025    Ulster            0.595103


In [6]:
line_df.to_csv("sari_yearly_equity.csv", index=False)


In [7]:
#i need to print out the list of data to add it on vega-lite manually
print(line_df.to_dict(orient="records"))

[{'year': 2023, 'province': 'Leinster', 'workshops_per_100k': 1.3447675749642325}, {'year': 2023, 'province': 'Munster', 'workshops_per_100k': 0.0}, {'year': 2023, 'province': 'Connacht', 'workshops_per_100k': 0.6434882209481155}, {'year': 2023, 'province': 'Ulster', 'workshops_per_100k': 2.380413953986598}, {'year': 2024, 'province': 'Leinster', 'workshops_per_100k': 0.9839762743640725}, {'year': 2024, 'province': 'Munster', 'workshops_per_100k': 0.9177882038517735}, {'year': 2024, 'province': 'Connacht', 'workshops_per_100k': 0.0}, {'year': 2024, 'province': 'Ulster', 'workshops_per_100k': 0.8926552327449743}, {'year': 2025, 'province': 'Leinster', 'workshops_per_100k': 2.295944640182836}, {'year': 2025, 'province': 'Munster', 'workshops_per_100k': 0.27533646115553206}, {'year': 2025, 'province': 'Connacht', 'workshops_per_100k': 0.16087205523702888}, {'year': 2025, 'province': 'Ulster', 'workshops_per_100k': 0.5951034884966495}]


In [8]:
monthly = master_df.groupby(["year", "month", "province"]).size().reset_index(name="workshops")

monthly["year_month"] = monthly["year"].astype(str) + "-" + monthly["month"].astype(str)

monthly = monthly.merge(province_population, on="province")

monthly["workshops_per_100k"] = (monthly["workshops"] / monthly["population"]) * 100000

monthly = monthly.sort_values(["year", "month"])

print(monthly.head())

   year  month  province  workshops year_month  population  workshops_per_100k
0  2023      1  Connacht          1     2023-1      621612            0.160872
1  2023      2  Leinster          2     2023-2     3048854            0.065598
2  2023      3    Ulster          1     2023-3      336076            0.297552
3  2023      4  Connacht          1     2023-4      621612            0.160872
4  2023      4    Ulster          1     2023-4      336076            0.297552


In [9]:
monthly["year_month"] = (
    monthly["year"].astype(str) + "-" +
    monthly["month"].astype(str).str.zfill(2)
)

In [10]:
monthly = monthly.sort_values(["year", "month"])

In [11]:
monthly_df = monthly[["year_month", "province", "workshops_per_100k"]]

monthly_df["workshops_per_100k"] = monthly_df["workshops_per_100k"].round(2)

monthly_df.to_csv("sari_monthly_equity.csv", index=False)

print(monthly_df.head())

  year_month  province  workshops_per_100k
0    2023-01  Connacht                0.16
1    2023-02  Leinster                0.07
2    2023-03    Ulster                0.30
3    2023-04  Connacht                0.16
4    2023-04    Ulster                0.30


C:\Users\New User\AppData\Local\Temp\ipykernel_24516\4260038279.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  monthly_df["workshops_per_100k"] = monthly_df["workshops_per_100k"].round(2)


In [12]:
monthly_df["province"].unique()

array(['Connacht', 'Leinster', 'Ulster', 'Munster'], dtype=object)

In [13]:
import pandas as pd

# create full list of provinces
provinces = ["Leinster", "Munster", "Connacht", "Ulster"]

# create full range of months in dataset
all_months = pd.date_range(
    start=master_df["date"].min(),
    end=master_df["date"].max(),
    freq="MS"
)

# convert to dataframe
calendar = pd.DataFrame({
    "year_month": all_months.strftime("%Y-%m")
})

# create province dataframe
province_df = pd.DataFrame({"province": provinces})

# cross join months and provinces
calendar["key"] = 1
province_df["key"] = 1
full_grid = calendar.merge(province_df, on="key").drop("key", axis=1)

# merge with your monthly data
monthly_df = full_grid.merge(
    monthly_df,
    on=["year_month", "province"],
    how="left"
)

# replace missing values with 0
monthly_df["workshops_per_100k"] = monthly_df["workshops_per_100k"].fillna(0)

# sort
monthly_df = monthly_df.sort_values(["year_month", "province"])

# preview
print(monthly_df.head(20))

   year_month  province  workshops_per_100k
2     2023-02  Connacht                0.00
0     2023-02  Leinster                0.07
1     2023-02   Munster                0.00
3     2023-02    Ulster                0.00
6     2023-03  Connacht                0.00
4     2023-03  Leinster                0.00
5     2023-03   Munster                0.00
7     2023-03    Ulster                0.30
10    2023-04  Connacht                0.16
8     2023-04  Leinster                0.00
9     2023-04   Munster                0.00
11    2023-04    Ulster                0.30
14    2023-05  Connacht                0.16
12    2023-05  Leinster                0.03
13    2023-05   Munster                0.00
15    2023-05    Ulster                0.00
18    2023-06  Connacht                0.16
16    2023-06  Leinster                0.03
17    2023-06   Munster                0.00
19    2023-06    Ulster                0.00


In [14]:
monthly_df.to_csv("sari_monthly_equity.csv", index=False)

In [15]:
monthly_df["year_month"] = pd.to_datetime(monthly_df["year_month"])

In [16]:
monthly_df["workshops_per_100k"] = monthly_df["workshops_per_100k"].round(2)

In [17]:
monthly_df.to_csv("sari_monthly_equity.csv", index=False)

In [18]:
monthly_df.describe()

,year_month,workshops_per_100k
count,140,140.000000
mean,2024-07-01 07:32:34.285714176,0.074000
min,2023-02-01 00:00:00,0.000000
25%,2023-10-01 00:00:00,0.000000
50%,2024-07-01 00:00:00,0.000000
75%,2025-04-01 00:00:00,0.090000
max,2025-12-01 00:00:00,1.190000
std,NaN,0.157086


In [19]:
#i need to print out the list of data to add it on vega-lite manually
print(monthly_df.to_dict(orient="records"))

[{'year_month': Timestamp('2023-02-01 00:00:00'), 'province': 'Connacht', 'workshops_per_100k': 0.0}, {'year_month': Timestamp('2023-02-01 00:00:00'), 'province': 'Leinster', 'workshops_per_100k': 0.07}, {'year_month': Timestamp('2023-02-01 00:00:00'), 'province': 'Munster', 'workshops_per_100k': 0.0}, {'year_month': Timestamp('2023-02-01 00:00:00'), 'province': 'Ulster', 'workshops_per_100k': 0.0}, {'year_month': Timestamp('2023-03-01 00:00:00'), 'province': 'Connacht', 'workshops_per_100k': 0.0}, {'year_month': Timestamp('2023-03-01 00:00:00'), 'province': 'Leinster', 'workshops_per_100k': 0.0}, {'year_month': Timestamp('2023-03-01 00:00:00'), 'province': 'Munster', 'workshops_per_100k': 0.0}, {'year_month': Timestamp('2023-03-01 00:00:00'), 'province': 'Ulster', 'workshops_per_100k': 0.3}, {'year_month': Timestamp('2023-04-01 00:00:00'), 'province': 'Connacht', 'workshops_per_100k': 0.16}, {'year_month': Timestamp('2023-04-01 00:00:00'), 'province': 'Leinster', 'workshops_per_100k':

In [27]:

primary = pd.read_excel("primary_schools.xlsx")
secondary = pd.read_excel("secondary_schools.xlsx")

schools = pd.concat([primary, secondary], ignore_index=True)

print(schools.head(200))
#print(primary.head())

    Unnamed: 0                                         Unnamed: 1  Unnamed: 2  \
0          NaN                                                NaN         NaN   
1          NaN                                                NaN         NaN   
2          NaN                      FINAL DATA as of 17 July 2024         NaN   
3          NaN                                                NaN         NaN   
4          NaN  Class Size Information 2024/2025 - Explanatory...         NaN   
5          NaN                                                NaN         NaN   
6          NaN  These worksheets contain class size informatio...         NaN   
7          NaN                                                NaN         NaN   
8          NaN  Worksheet 1 (1 Class Size Range) contains a su...         NaN   
9          NaN                                                NaN         NaN   
10         NaN  Worksheet 2 (2 Individual Class Data) contains...         NaN   
11         NaN              

In [28]:
import pandas as pd

xls = pd.ExcelFile("primary_schools.xlsx")
print(xls.sheet_names)

['Explanatory Note', '1 Class Size Range', '2 Individual Class Data']


In [29]:
primary = pd.read_excel("primary_schools.xlsx", sheet_name=1)
print(primary.head())

  Class Size Range                 Unnamed: 1  \
0      Roll Number  Academic Year (Enrolment)   
1           00359V                       2024   
2           00373P                       2024   
3           00467B                       2024   
4           00512D                       2024   

                        Unnamed: 2        Unnamed: 3        Unnamed: 4  \
0                    Official Name  Address (Line 1)  Address (Line 2)   
1  ST. LOUIS GIRLS NATIONAL SCHOOL         Park Road          Monaghan   
2          DERAVOY NATIONAL SCHOOL           Deravoy           Emyvale   
3                BALLINSPITTLE N S       Ballycatten     Ballinspittle   
4             MIDLETON CONVENT N S          Midleton          Co. Cork   

           Unnamed: 5 Unnamed: 6                   Unnamed: 7 Unnamed: 8  \
0  County Description    Eircode  Local Authority Description       DEIS   
1            Monaghan    H18HK31      Monaghan County Council          Y   
2            Monaghan    H18PY11

In [42]:
primary = pd.read_excel("primary_schools.xlsx", sheet_name=1, skiprows=1)


print(primary.columns)

Index(['Roll Number', 'Academic Year (Enrolment)', 'Official Name',
       'Address (Line 1)', 'Address (Line 2)', 'County Description', 'Eircode',
       'Local Authority Description', 'DEIS', 'Ethos', 'Count 0-9',
       'Count 10-19', 'Count 20-24', 'Count 25-29', 'Count 30-34',
       'Count 35-39', 'Count over 40', 'Enrolment per Return', 'Count Class'],
      dtype='object')


In [43]:
primary = primary.rename(columns={"County Description": "county"})

In [44]:

print(primary.columns)

Index(['Roll Number', 'Academic Year (Enrolment)', 'Official Name',
       'Address (Line 1)', 'Address (Line 2)', 'county', 'Eircode',
       'Local Authority Description', 'DEIS', 'Ethos', 'Count 0-9',
       'Count 10-19', 'Count 20-24', 'Count 25-29', 'Count 30-34',
       'Count 35-39', 'Count over 40', 'Enrolment per Return', 'Count Class'],
      dtype='object')


In [52]:
secondary = pd.read_excel("secondary_schools.xlsx", sheet_name="School List", skiprows=0)

print(secondary.head())
print(secondary.columns)

   Academic Year Roll Number       Official School Name          Address 1  \
0         2024.0      60010P    Loreto Secondary School         Brick Lane   
1         2024.0      60021U  St Marys Secondary School        Main Street   
2         2024.0      60030V          Blackrock College  Blackrock College   
3         2024.0      60040B         Willow Park School          Rock Road   
4         2024.0      60041D              Coláiste Eoin  Baile an Bhóthair   

              Address 2        Address 3 Address 4  Eircode  School Latitude  \
0            Balbriggan        Co Dublin       NaN  K32R248        53.612259   
1              Baldoyle        Dublin 13       NaN  D13W208        53.396912   
2             Rock Road       Co. Dublin       NaN  A94FK84        53.303651   
3             Blackrock        Co Dublin       NaN  A94TW98        53.306045   
4  Bóthair Stigh Lorgan  Co. Átha Cliath       NaN  A94E122        53.302626   

   School Longitude  ...           Irish Classific

In [53]:
secondary["county"] = secondary["Address 3"].str.replace("Co.", "", regex=False)
secondary["county"] = secondary["county"].str.replace("Co", "", regex=False)
secondary["county"] = secondary["county"].str.strip()

print(secondary[["Official School Name","county"]].head())

        Official School Name       county
0    Loreto Secondary School       Dublin
1  St Marys Secondary School    Dublin 13
2          Blackrock College       Dublin
3         Willow Park School       Dublin
4              Coláiste Eoin  Átha Cliath


In [58]:
secondary["county"] = secondary["Address 3"].str.replace("Co.", "", regex=False)
secondary["county"] = secondary["county"].str.replace("Co", "", regex=False)
secondary["county"] = secondary["county"].str.strip()

# fix Irish name
secondary["county"] = secondary["county"].replace({
    "Átha Cliath": "Dublin",
    "Atha Cliath": "Dublin"
})

secondary["county"] = secondary["county"].str.replace(r"Dublin\s*\d+", "Dublin", regex=True)
print(secondary["county"].unique())

['Dublin' 'An Charraig Dhubh' nan 'Blackrock' 'Rathfarnham'
 'Baile Átha Cliath 7' 'DublinW' 'Dublinw' 'Rathgar' 'Ath Cliath 9'
 'Glasnevin' 'Warrenmount' 'Swords  Dublin' 'olock' 'Cavan' 'Carlow'
 'Kerry' 'Chiarraí' 'Kilkenny' 'Kildare.' 'Kildare' 'Wicklow' 'Bray'
 'Clare' 'rk' 'rk.' 'rcaigh' 'Donegal' 'Galway' 'na Gaillimhe' 'Westmeath'
 'Mullingar' 'Laois' 'Portlaoise' 'Wexford' 'Wexford.' 'Y35 XV70'
 'Longford' 'Louth' 'Lú' 'Dundalk' 'Limerick' 'Meath' 'Mayo'
 'Clar Cloinne Mhuiris' 'Monaghan' 'Waterford' 'Roscommon Town'
 'Roscommon' 'Sligo' 'Tipperary' 'Thurles' 'Tipperary.' 'Tipperary Town'
 'Offaly' 'Rathmichael' 'Port Láirge' 'Inis Córthaidh' 'Ráth Fearnáin'
 'Balbriggan' 'Firhouse' 'Drinan' 'Towlerton, Castletroy' 'Tuam'
 'Clondalkin' 'Baile Atha Cliath 22' 'Templeogue' 'Maynooth'
 'Chill Mhantain' 'Scariff' 'Chorcaí' 'Lifford' 'Leifear' 'Dhún na nGall'
 'Oileain Árann' 'Gaillimh' 'na.Gaillimhe' 'Enniscorthy' 'na Mí' 'Ballina'
 'Phort Láirge' 'BUTTEVANT' 'Skerries' 'Doirí Bea

In [59]:
import re

counties = [
"Carlow","Cavan","Clare","Cork","Donegal","Dublin","Galway","Kerry",
"Kildare","Kilkenny","Laois","Leitrim","Limerick","Longford","Louth",
"Mayo","Meath","Monaghan","Offaly","Roscommon","Sligo","Tipperary",
"Waterford","Westmeath","Wexford","Wicklow"
]

def extract_county(text):
    if pd.isna(text):
        return None
    text = str(text).lower()
    for c in counties:
        if c.lower() in text:
            return c
    return None

secondary["county"] = secondary["Address 3"].apply(extract_county)

print(secondary["county"].unique())

['Dublin' None 'Cavan' 'Carlow' 'Kerry' 'Kilkenny' 'Kildare' 'Wicklow'
 'Clare' 'Cork' 'Donegal' 'Galway' 'Meath' 'Laois' 'Wexford' 'Longford'
 'Louth' 'Limerick' 'Mayo' 'Monaghan' 'Waterford' 'Roscommon' 'Sligo'
 'Tipperary' 'Offaly']


In [60]:
secondary = secondary.dropna(subset=["county"])

In [61]:
schools = pd.concat([primary, secondary], ignore_index=True)

In [62]:
schools_by_county = schools.groupby("county").size().reset_index(name="schools")

print(schools_by_county)

       county  schools
0      Carlow       47
1       Cavan       80
2       Clare      117
3        Cork      374
4     Donegal      185
5      Dublin      564
6      Galway      234
7       Kerry      147
8     Kildare      115
9    Kilkenny       72
10      Laois       68
11    Leitrim       37
12   Limerick      144
13   Longford       38
14      Louth       84
15       Mayo      162
16      Meath      135
17   Monaghan       66
18     Offaly       74
19  Roscommon       93
20      Sligo       65
21  Tipperary      170
22  Waterford       81
23  Westmeath       73
24    Wexford      114
25    Wicklow       97


In [63]:
#county to province
province_map = {
"Carlow":"Leinster","Dublin":"Leinster","Kildare":"Leinster","Kilkenny":"Leinster",
"Laois":"Leinster","Longford":"Leinster","Louth":"Leinster","Meath":"Leinster",
"Offaly":"Leinster","Westmeath":"Leinster","Wexford":"Leinster","Wicklow":"Leinster",

"Cork":"Munster","Kerry":"Munster","Clare":"Munster","Limerick":"Munster",
"Tipperary":"Munster","Waterford":"Munster",

"Galway":"Connacht","Mayo":"Connacht","Roscommon":"Connacht","Sligo":"Connacht","Leitrim":"Connacht",

"Donegal":"Ulster","Cavan":"Ulster","Monaghan":"Ulster"
}

schools_by_county["province"] = schools_by_county["county"].map(province_map)

In [64]:
schools_by_province = schools_by_county.groupby("province")["schools"].sum().reset_index()

print(schools_by_province)

   province  schools
0  Connacht      591
1  Leinster     1481
2   Munster     1033
3    Ulster      331


In [66]:
#merge with workshop data

equity = workshops_by_province.merge(schools_by_province, on="province")

#then we compute the metric
equity["workshops_per_100_schools"] = (equity["workshops"] / equity["schools"]) * 100

print(equity)

   province  workshops  schools  workshops_per_100_schools
0  Connacht          5      591                   0.846024
1  Leinster        141     1481                   9.520594
2   Munster         13     1033                   1.258470
3    Ulster         13      331                   3.927492


In [67]:
#then we save the file for vega

equity.to_csv("sari_workshops_per_100_schools.csv", index=False)